# A7 — Clasificación binaria CONTR con Gemini API (Tarea 2)

Clasificación binaria: fragmento **CONTR** (contribución científica explícita) vs. **no-CONTR**.

- Dataset: `DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL.xlsx`  
- Evaluación sobre partición `dataset_type == EVAL` (validación humana)
- Estrategia: zero-shot (sin ejemplos en el prompt, sin patrones de selección)
- Modelo: Gemini via `google-genai` SDK  
- Pipeline: async con semáforo + checkpoint cada 50 requests  
- Métricas: Precision, Recall, F1, Accuracy, ROC-AUC  
- Registro de tokens y latencia para análisis A8

In [ ]:
!pip install -q google-genai pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Configuración

In [ ]:
import asyncio
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google import genai
from google.genai import types
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

DRIVE_ROOT  = Path("/content/drive/MyDrive")
DATA_PATH   = DRIVE_ROOT / "DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL_v3.xlsx"
RESULTS_DIR = DRIVE_ROOT / "results_a7_task2_gemini"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gemini-2.5-flash"

COST_INPUT_PER_M  = 0.15
COST_OUTPUT_PER_M = 0.60

TEMPERATURE    = 0
MAX_TOKENS     = 60
MAX_TOKENS_COT = 350
MAX_TEXT_WORDS = 700

# gemini-2.5-flash: 1000 RPM → 15 requests simultáneos es margen seguro
CONCURRENCY = 15

api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

if not api_key:
    raise EnvironmentError("GOOGLE_API_KEY no encontrada. Agregar en Colab Secrets.")

client = genai.Client(api_key=api_key)

print(f"Modelo:       {MODEL}")
print(f"Dataset:      {DATA_PATH}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Concurrency:  {CONCURRENCY}")

## 2. Carga y exploración del dataset

In [ ]:
df_raw = pd.read_excel(DATA_PATH, index_col=0)

df_raw["LABEL_T2"]       = (df_raw["label"] == "CONTR").astype(int)
df_raw["TIPO_DATASET_T2"] = df_raw["dataset_type"]

df = df_raw.copy()
print(f"Shape total: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print()
print("Distribución dataset_type:")
print(df["TIPO_DATASET_T2"].value_counts())
print()
print("Distribución LABEL_T2 (total):")
print(df["LABEL_T2"].value_counts())

df_eval = df[df["TIPO_DATASET_T2"] == "EVAL"].reset_index(drop=True)
print()
print(f"Filas de evaluación (EVAL): {len(df_eval)}")
print("LABEL_T2 en EVAL:")
print(df_eval["LABEL_T2"].value_counts())
print()
print("Composición retórica (label) en EVAL:")
print(df_eval.groupby(["LABEL_T2", "label"]).size().reset_index(name="n").to_string(index=False))


In [ ]:
df_eval["word_count"] = df_eval["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label_val, color in [(axes[0], 1, "steelblue"), (axes[1], 0, "coral")]:
    subset = df_eval[df_eval["LABEL_T2"] == label_val]["word_count"]
    ax.hist(subset, bins=40, color=color, edgecolor="white")
    ax.set_title(f"{'CONTR (1)' if label_val == 1 else 'No-CONTR (0)'}  —  n={len(subset)}")
    ax.set_xlabel("Palabras")
    ax.set_ylabel("Frecuencia")
    ax.axvline(subset.median(), color="black", linestyle="--", label=f"mediana={subset.median():.0f}")
    ax.legend()

plt.suptitle("Longitud de fragmentos en EVAL — por clase", y=1.02)
plt.tight_layout()
plt.show()
print(df_eval.groupby("LABEL_T2")["word_count"].describe().round(1))

## 3. Diseño del prompt (zero-shot)

In [ ]:
SYSTEM_PROMPT = """Eres un experto en análisis del discurso científico en español.
Tu tarea es determinar si un fragmento de un artículo científico declara explícitamente una CONTRIBUCIÓN CIENTÍFICA ORIGINAL.

Un fragmento ES una contribución científica (label=1) si y solo si:
- Declara de forma explícita los aportes originales del trabajo.
- Contiene expresiones como: "proponemos", "nuestra contribución es", "presentamos un nuevo método/sistema/modelo",
  "el aporte principal de este trabajo es", "a diferencia de trabajos previos, nuestro enfoque",
  "contribuimos con", "nuestro trabajo introduce", "en este artículo proponemos".
- Hace una declaración directa de lo nuevo que el trabajo aporta al campo.

Un fragmento NO ES una contribución (label=0) si:
- Solo presenta resultados obtenidos sin declarar que son aportes originales.
- Solo interpreta o discute resultados, aunque los compare con otros trabajos.
- Solo presenta los objetivos del trabajo sin declararlos como aporte.
- Describe la metodología usada.
- Es revisión de literatura o trabajo previo de otros autores.
- Resume conclusiones sin declarar explícitamente aportes.
- Habla de limitaciones o trabajo futuro.

Responde ÚNICAMENTE con un JSON válido en el formato exacto:
{"label": <0 o 1>, "confidence": <número entre 0.0 y 1.0>}

Donde label=1 significa que el fragmento SÍ es una contribución científica explícita,
y label=0 significa que NO lo es.

No incluyas explicaciones, markdown ni texto fuera del JSON."""


GEN_CONFIG_ZEROSHOT = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=TEMPERATURE,
    max_output_tokens=MAX_TOKENS,
)

SYSTEM_PROMPT_COT = """Eres un experto en análisis del discurso científico en español.
Tu tarea es determinar si un fragmento de un artículo científico declara explícitamente una CONTRIBUCIÓN CIENTÍFICA ORIGINAL.

Razona brevemente paso a paso:
1. ¿Contiene el fragmento señales explícitas de aporte original? (verbos en primera persona del plural: \"proponemos\", \"presentamos\", \"desarrollamos\"; frases como \"nuestra contribución es\", \"el aporte principal de este trabajo\", \"a diferencia de trabajos previos, nuestro enfoque\")
2. ¿Son señales verdaderas de contribución o describen resultados, metodología, revisión de literatura, objetivos o conclusiones?
3. Conclusión: label=1 si hay declaración explícita de aporte original, label=0 en caso contrario.

Escribe tu razonamiento en 1-2 oraciones y en la última línea responde únicamente con JSON válido:
{\"label\": <0 o 1>, \"confidence\": <número entre 0.0 y 1.0>}"""

GEN_CONFIG_COT = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT_COT,
    temperature=TEMPERATURE,
    max_output_tokens=MAX_TOKENS_COT,
)


def build_user_message(text: str, max_words: int = MAX_TEXT_WORDS) -> str:
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words])
    return f"Determina si el siguiente fragmento de un artículo científico en español es una contribución científica explícita:\n\n{text}"


GEN_CONFIG = GEN_CONFIG_ZEROSHOT  # alias for backward compat
print("Prompts listos: zero-shot, CoT.")
print(f"Longitud del system prompt: {len(SYSTEM_PROMPT.split())} palabras")

## 4. Pipeline de inferencia

In [ ]:
def parse_response(raw: str) -> dict | None:
    raw = re.sub(r"```(?:json)?\n?", "", raw).replace("```", "").strip()
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group())
        label = parsed.get("label")
        if label not in (0, 1, "0", "1"):
            return None
        return {"label": int(label), "confidence": float(parsed.get("confidence", 0.0))}
    except (json.JSONDecodeError, ValueError, TypeError):
        return None



def parse_response_cot(raw: str) -> dict | None:
    raw = re.sub(r"```(?:json)?\n?", "", raw).replace("```", "").strip()
    matches = list(re.finditer(r"\{[^{}]*\}", raw))
    for m in reversed(matches):
        try:
            parsed = json.loads(m.group())
            label = parsed.get("label")
            if label not in (0, 1, "0", "1"):
                continue
            return {"label": int(label), "confidence": float(parsed.get("confidence", 0.0))}
        except (json.JSONDecodeError, ValueError, TypeError):
            continue
    return None


async def classify_async(semaphore, idx, row):
    async with semaphore:
        content = build_user_message(row["text"])
        result = {
            "label": None, "confidence": None,
            "tokens_in": 0, "tokens_out": 0,
            "latency_s": 0.0, "error": None,
            "_idx": idx,
            "true_label": int(row["LABEL_T2"]),
            "doc_id": row["doc_id"],
            "orig_label": row["label"],
        }
        for attempt in range(3):
            try:
                t0 = time.time()
                response = await client.aio.models.generate_content(
                    model=MODEL,
                    contents=content,
                    config=GEN_CONFIG,
                )
                result["latency_s"] = time.time() - t0
                if response.usage_metadata:
                    result["tokens_in"]  = response.usage_metadata.prompt_token_count or 0
                    result["tokens_out"] = response.usage_metadata.candidates_token_count or 0
                parsed = parse_response(response.text.strip())
                if parsed:
                    result["label"]      = parsed["label"]
                    result["confidence"] = parsed["confidence"]
                    return result
                else:
                    result["error"] = f"parse_fail: {response.text[:80]}"
            except Exception as e:
                err = str(e)
                result["error"] = err[:120]
                await asyncio.sleep(10 if "429" in err else 2 ** attempt)
        return result



async def classify_async_cot(semaphore, idx, row):
    async with semaphore:
        content = build_user_message(row["text"])
        result = {
            "label": None, "confidence": None,
            "tokens_in": 0, "tokens_out": 0,
            "latency_s": 0.0, "error": None,
            "_idx": idx,
            "true_label": int(row["LABEL_T2"]),
            "doc_id": row["doc_id"],
            "orig_label": row["label"],
        }
        for attempt in range(3):
            try:
                t0 = time.time()
                response = await client.aio.models.generate_content(
                    model=MODEL,
                    contents=content,
                    config=GEN_CONFIG_COT,
                )
                result["latency_s"] = time.time() - t0
                if response.usage_metadata:
                    result["tokens_in"]  = response.usage_metadata.prompt_token_count or 0
                    result["tokens_out"] = response.usage_metadata.candidates_token_count or 0
                parsed = parse_response_cot(response.text.strip())
                if parsed:
                    result["label"]      = parsed["label"]
                    result["confidence"] = parsed["confidence"]
                    return result
                else:
                    result["error"] = f"parse_fail: {response.text[:80]}"
            except Exception as e:
                err = str(e)
                result["error"] = err[:120]
                await asyncio.sleep(10 if "429" in err else 2 ** attempt)
        return result


async def run_batch_async(df, output_path, classify_fn=None, desc="clasificando"):
    output_path = Path(output_path)

    if output_path.exists():
        done     = pd.read_csv(output_path)
        done_ids = set(done["_idx"].tolist())
        print(f"Checkpoint: {len(done_ids)} ya procesados de {len(df)}.")
    else:
        done     = pd.DataFrame()
        done_ids = set()

    pending = [(idx, row) for idx, row in df.iterrows() if idx not in done_ids]
    print(f"Pendientes: {len(pending)}")

    if not pending:
        return done

    if classify_fn is None:
        classify_fn = classify_async

    semaphore   = asyncio.Semaphore(CONCURRENCY)
    all_results = list(done.to_dict("records")) if not done.empty else []

    tasks = [
        classify_fn(semaphore, idx, row)
        for idx, row in pending
    ]

    completed = 0
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=desc):
        result = await coro
        all_results.append(result)
        completed += 1
        if completed % 50 == 0:
            pd.DataFrame(all_results).to_csv(output_path, index=False)

    combined = pd.DataFrame(all_results)
    combined.to_csv(output_path, index=False)
    print(f"Guardado: {output_path} — {len(combined)} filas")
    return combined


print("Pipeline listo. Funciones: classify_async (zero-shot), classify_async_fewshot (few-shot), classify_async_cot (CoT)")

## 5. Test de conectividad

In [ ]:
async def test_single():
    row = df_eval.iloc[0]
    content = build_user_message(row["text"])
    try:
        response = await client.aio.models.generate_content(
            model=MODEL,
            contents=content,
            config=GEN_CONFIG,
        )
        print("Status: OK")
        print(f"Etiqueta real (LABEL_T2): {row['LABEL_T2']}  |  Etiqueta retórica: {row['label']}")
        print(f"Texto: {row['text'][:200]}")
        print(f"Respuesta: {response.text}")
        print(f"Parsed: {parse_response(response.text)}")
        print(f"Usage: {response.usage_metadata}")
    except Exception as e:
        print("ERROR:", e)

await test_single()

## 6. Ejecución del clasificador

In [ ]:
PREDS_PATH = RESULTS_DIR / "predictions_a7_gemini_eval.csv"

print(f"Filas a clasificar (EVAL): {len(df_eval)}")
print(f"Tiempo estimado ({CONCURRENCY} concurrentes, ~2s/lote): ~{len(df_eval) / CONCURRENCY * 2 / 60:.0f} min")
print()

preds = await run_batch_async(df_eval, PREDS_PATH)
print(f"\nFallidos: {preds['label'].isna().sum()} / {len(preds)}")

## 7. Métricas de clasificación

In [ ]:
valid    = preds.dropna(subset=["label"]).copy()
n_failed = len(preds) - len(valid)

y_true = valid["true_label"].astype(int).tolist()
y_pred = valid["label"].astype(int).tolist()
y_conf = valid["confidence"].fillna(0.5).tolist()

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
f1_macro  = f1_score(y_true, y_pred, average="macro", zero_division=0)
try:
    auc = roc_auc_score(y_true, y_conf)
except Exception:
    auc = float("nan")

print(f"{'='*50}")
print(f"Modelo: {MODEL}  |  Estrategia: zero-shot")
print(f"{'='*50}")
print(f"Requests válidos:  {len(valid)} / {len(preds)}")
print(f"Requests fallidos: {n_failed}")
print()
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}  (CONTR=1)")
print(f"Recall:      {recall:.4f}  (CONTR=1)")
print(f"F1 (CONTR):  {f1:.4f}")
print(f"F1 Macro:    {f1_macro:.4f}")
print(f"ROC-AUC:     {auc:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["No-CONTR (0)", "CONTR (1)"], zero_division=0))

## 8. Matriz de confusión

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      "d",   "Conteos absolutos"),
    (axes[1], cm_norm, ".2f", "Proporciones por fila"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=["No-CONTR (0)", "CONTR (1)"],
        yticklabels=["No-CONTR (0)", "CONTR (1)"],
        vmin=0, ax=ax, linewidths=0.3,
    )
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    ax.set_title(f"{title}  |  F1={f1:.3f}  |  Acc={accuracy:.3f}")

plt.suptitle(f"Tarea 2 — {MODEL} (zero-shot)", y=1.02, fontsize=13)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix_a7_gemini.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Análisis de errores por etiqueta retórica

In [ ]:
errors = valid[valid["true_label"] != valid["label"]].copy()
fn = errors[(errors["true_label"] == 1) & (errors["label"] == 0)]
fp = errors[(errors["true_label"] == 0) & (errors["label"] == 1)]

print(f"Errores totales: {len(errors)} / {len(valid)} ({len(errors)/len(valid)*100:.1f}%)")
print(f"Falsos Negativos (CONTR predicho como No-CONTR): {len(fn)}")
print(f"Falsos Positivos (No-CONTR predicho como CONTR): {len(fp)}")
print()
print("Tasa de error por etiqueta retórica original (orig_label):")
err_by_label = (
    errors.groupby("orig_label").size() / valid.groupby("orig_label").size()
).sort_values(ascending=False).round(3)
print(err_by_label.to_string())

In [ ]:
print("=== FALSOS NEGATIVOS (real=CONTR, pred=No-CONTR) ===\n")
sample_fn = fn.sample(min(5, len(fn)), random_state=42)
for i, (_, row) in enumerate(sample_fn.iterrows(), 1):
    orig = df_eval.iloc[row["_idx"]]
    print(f"--- Ej. {i} | Conf: {row['confidence']:.2f} | Etiqueta retórica: {orig['label']} ---")
    print(orig["text"][:400])
    print()

In [ ]:
print("=== FALSOS POSITIVOS (real=No-CONTR, pred=CONTR) ===\n")
sample_fp = fp.sample(min(5, len(fp)), random_state=42)
for i, (_, row) in enumerate(sample_fp.iterrows(), 1):
    orig = df_eval.iloc[row["_idx"]]
    print(f"--- Ej. {i} | Conf: {row['confidence']:.2f} | Etiqueta retórica: {orig['label']} ---")
    print(orig["text"][:400])
    print()

## 10. Costo y latencia (A8)

In [ ]:
total_in  = preds["tokens_in"].sum()
total_out = preds["tokens_out"].sum()
cost_usd  = (total_in / 1e6) * COST_INPUT_PER_M + (total_out / 1e6) * COST_OUTPUT_PER_M
lat       = preds["latency_s"]

print(f"{'='*50}")
print(f"Modelo: {MODEL}")
print(f"Tokens entrada:    {total_in:,}")
print(f"Tokens salida:     {total_out:,}")
print(f"Costo USD:         ${cost_usd:.4f}")
print(f"Costo/1000 docs:   ${cost_usd / len(preds) * 1000:.4f} USD")
print(f"Latencia media:    {lat.mean():.2f} s")
print(f"Latencia p95:      {lat.quantile(0.95):.2f} s")
print(f"{'='*50}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lat, bins=40, color="steelblue", edgecolor="white")
ax.axvline(lat.mean(), color="red", linestyle="--", label=f"media={lat.mean():.2f}s")
ax.axvline(lat.quantile(0.95), color="orange", linestyle="--", label=f"p95={lat.quantile(0.95):.2f}s")
ax.set_title(f"Latencia — {MODEL} (zero-shot)")
ax.set_xlabel("Latencia (s)")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS_DIR / "latency_a7_gemini.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Resumen y guardado

In [ ]:
summary = {
    "modelo":            MODEL,
    "estrategia":        "zero-shot",
    "particion_eval":    "EVAL",
    "n_eval":            len(preds),
    "n_validos":         len(valid),
    "n_fallidos":        n_failed,
    "accuracy":          round(accuracy, 4),
    "precision_contr":   round(precision, 4),
    "recall_contr":      round(recall, 4),
    "f1_contr":          round(f1, 4),
    "f1_macro":          round(f1_macro, 4),
    "roc_auc":           round(auc, 4) if not np.isnan(auc) else None,
    "tokens_in":         int(total_in),
    "tokens_out":        int(total_out),
    "costo_usd":         round(cost_usd, 4),
    "latencia_media_s":  round(lat.mean(), 2),
    "latencia_p95_s":    round(lat.quantile(0.95), 2),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULTS_DIR / "summary_a7_gemini.csv", index=False)

print("Resumen guardado en:", RESULTS_DIR / "summary_a7_gemini.csv")
print()
print(summary_df.T.to_string(header=False))

## 12. Few-shot y CoT

Dos estrategias adicionales sobre el mismo conjunto EVAL, sin re-ejecutar los resultados zero-shot.

In [ ]:
FEWSHOT_EXAMPLES = (
    df_train[df_train["LABEL_T2"] == 1]
    .sample(3, random_state=42)[["text", "LABEL_T2"]]
    .rename(columns={"LABEL_T2": "label"})
    .to_dict("records")
) + (
    df_train[df_train["LABEL_T2"] == 0]
    .sample(3, random_state=42)[["text", "LABEL_T2"]]
    .rename(columns={"LABEL_T2": "label"})
    .to_dict("records")
)

def build_fewshot_contents(text: str) -> list:
    contents = []
    for ex in FEWSHOT_EXAMPLES:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=build_user_message(ex["text"]))]
        ))
        contents.append(types.Content(
            role="model",
            parts=[types.Part(text=json.dumps({"label": ex["label"], "confidence": 1.0}))]
        ))
    contents.append(types.Content(
        role="user",
        parts=[types.Part(text=build_user_message(text))]
    ))
    return contents

async def classify_async_fewshot(semaphore, idx, row):
    async with semaphore:
        content = build_fewshot_contents(row["text"])
        result = {
            "label": None, "confidence": None,
            "tokens_in": 0, "tokens_out": 0,
            "latency_s": 0.0, "error": None,
            "_idx": idx,
            "true_label": int(row["LABEL_T2"]),
            "doc_id": row["doc_id"],
            "orig_label": row["label"],
        }
        for attempt in range(3):
            try:
                t0 = time.time()
                response = await client.aio.models.generate_content(
                    model=MODEL,
                    contents=content,
                    config=GEN_CONFIG_ZEROSHOT,
                )
                result["latency_s"] = time.time() - t0
                if response.usage_metadata:
                    result["tokens_in"]  = response.usage_metadata.prompt_token_count or 0
                    result["tokens_out"] = response.usage_metadata.candidates_token_count or 0
                parsed = parse_response(response.text.strip())
                if parsed:
                    result["label"]      = parsed["label"]
                    result["confidence"] = parsed["confidence"]
                    return result
                else:
                    result["error"] = f"parse_fail: {response.text[:80]}"
            except Exception as e:
                err = str(e)
                result["error"] = err[:120]
                await asyncio.sleep(10 if "429" in err else 2 ** attempt)
        return result

n_pos = sum(e["label"] for e in FEWSHOT_EXAMPLES)
print(f"Few-shot examples: {len(FEWSHOT_EXAMPLES)} ({n_pos} CONTR=1, {len(FEWSHOT_EXAMPLES)-n_pos} no-CONTR=0)")
print(f"Turns por request: {len(build_fewshot_contents('test'))}")

### 12a. Few-shot (k=6, temp=0)

In [ ]:
PREDS_FEWSHOT_PATH = RESULTS_DIR / "predictions_a7_fewshot_eval.csv"

preds_fs = await run_batch_async(
    df_eval,
    PREDS_FEWSHOT_PATH,
    classify_fn=classify_async_fewshot,
    desc="few-shot",
)
print(f"Fallidos: {preds_fs['label'].isna().sum()} / {len(preds_fs)}")

### 12b. CoT (zero-shot + razonamiento)

In [ ]:
PREDS_COT_PATH = RESULTS_DIR / "predictions_a7_cot_eval.csv"

preds_cot = await run_batch_async(
    df_eval,
    PREDS_COT_PATH,
    classify_fn=classify_async_cot,
    desc="CoT",
)
print(f"Fallidos: {preds_cot['label'].isna().sum()} / {len(preds_cot)}")

### 12c. Comparación de estrategias

In [ ]:
def strategy_metrics(df_preds, strategy_name):
    valid = df_preds.dropna(subset=["label"]).copy()
    y_true = valid["true_label"].astype(int).tolist()
    y_pred = valid["label"].astype(int).tolist()
    y_conf = valid["confidence"].fillna(0.5).tolist()
    try:
        auc = roc_auc_score(y_true, y_conf)
    except Exception:
        auc = float("nan")
    cost = (df_preds["tokens_in"].sum() / 1e6) * COST_INPUT_PER_M +            (df_preds["tokens_out"].sum() / 1e6) * COST_OUTPUT_PER_M
    return {
        "Estrategia":    strategy_name,
        "N eval":        len(df_preds),
        "Válidos":       len(valid),
        "Fallidos":      len(df_preds) - len(valid),
        "Accuracy":      round(accuracy_score(y_true, y_pred), 4),
        "Precision":     round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall":        round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1 (CONTR)":    round(f1_score(y_true, y_pred, zero_division=0), 4),
        "F1 Macro":      round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "ROC-AUC":       round(auc, 4),
        "Costo USD":     round(cost, 4),
        "Latencia p50 s": round(df_preds["latency_s"].median(), 2),
        "Latencia p95 s": round(df_preds["latency_s"].quantile(0.95), 2),
    }

preds_zs = pd.read_csv(RESULTS_DIR / "predictions_a7_gemini_eval.csv")

rows = [
    strategy_metrics(preds_zs,  "Zero-shot"),
    strategy_metrics(preds_fs,  "Few-shot k=6"),
    strategy_metrics(preds_cot, "CoT"),
]
comparison = pd.DataFrame(rows).set_index("Estrategia")
print(comparison.to_string())

winner = comparison["F1 (CONTR)"].idxmax()
print(f"\nGanador por F1 (CONTR): {winner} ({comparison.loc[winner, 'F1 (CONTR)']:.4f})")

comparison.reset_index().to_csv(RESULTS_DIR / "summary_a7_comparison.csv", index=False)
print("Guardado: summary_a7_comparison.csv")